<a href="https://colab.research.google.com/github/sulucay01/multimodal-rs-segmentation/blob/dev/notebooks/04_multimodal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 — Multimodal experiments

DI725 Term Project — Multimodal Fusion for Remote Sensing Land Cover Segmentation.

This notebook trains text-augmented variants of UNetFormer (UNetFormer + frozen CLIP via cross-attention) and addresses the project's three research questions:

- **RQ1**: Does textual information improve segmentation over the vision-only baseline?
- **Sub-question 1 (caption-source ablation)**: Which caption type (hybrid, text-only, vision-only) gives the largest improvement?
- **Sub-question 2 (auxiliary supervision)**: Does adding a KL-divergence loss between predicted and ground-truth class distributions help on top of the best caption type?

The vision-only baseline (UNetFormer, no text) is established in `03_baselines.ipynb`. Multimodal experiments here use the same training setup (30 epochs, AdamW lr=6e-4, weighted CE + 0.4 × aux head loss, batch size 8, no augmentation) and only differ in the addition of text fusion.

Sections:
1. Setup
2. Dataset and DataLoaders (with caption)
3. UNetFormer architecture
4. Multimodal architecture (CLIP + cross-attention)
5. Training infrastructure
6. Caption-source ablation (RQ1 + Sub1)
7. Auxiliary KL-divergence loss (Sub2)
8. Test set evaluation

## 1. Setup

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Install dependencies. open_clip_torch provides the frozen CLIP text encoder.
!pip install transformers timm einops wandb open_clip_torch -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.1 MB/s eta 0:00:00


In [3]:
# Project paths (Drive) and a local working copy for fast I/O during training
from pathlib import Path
import shutil, os

PROJECT_DIR     = Path('/content/drive/MyDrive/DI725_Project')
DATA_DIR_DRIVE  = PROJECT_DIR / 'data'
SPLITS_CSV      = DATA_DIR_DRIVE / 'captions_with_splits.csv'
CHECKPOINTS_DIR = PROJECT_DIR / 'checkpoints'
RESULTS_DIR     = PROJECT_DIR / 'results'
CHECKPOINTS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

# Local SSD copy avoids Drive I/O bottleneck during training
LOCAL_DATA  = Path('/content/data')
LOCAL_IMAGES      = LOCAL_DATA / 'images'
LOCAL_MASKS_CLASS = LOCAL_DATA / 'masks_class'

if not LOCAL_DATA.exists():
    print('Copying data to local SSD...')
    shutil.copytree(DATA_DIR_DRIVE / 'images', LOCAL_IMAGES)
    shutil.copytree(DATA_DIR_DRIVE / 'masks_class', LOCAL_MASKS_CLASS)
    print('Done.')
else:
    print('Local data already present.')

print(f'Images: {LOCAL_IMAGES}')
print(f'Masks : {LOCAL_MASKS_CLASS}')

Copying data to local SSD...
Done.
Images: /content/data/images
Masks : /content/data/masks_class


In [4]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import time
import json
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

Device: cuda


In [5]:
# Weights & Biases setup for experiment tracking
import wandb
wandb.login()

WANDB_PROJECT = 'multimodal-rs-segmentation'

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: selingoktas98 (selingoktas98-metu-middle-east-technical-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [6]:
# Class definitions
CLASS_NAMES = ['Tree', 'Shrub', 'Grass', 'Crop', 'Built-up', 'Barren', 'Water']
NUM_CLASSES = len(CLASS_NAMES)

# 5 caption variants from the dataset
CAPTION_COLS = [
    'hybrid_gemma3-4b',
    'hybrid_qwen3-vl-8b',
    'text_qwen3-4b',
    'vision_gemma3-4b',
    'vision_qwen3-vl-8b',
]

In [7]:
# Load split CSV from Drive
split_df = pd.read_csv(SPLITS_CSV)
train_df = split_df[split_df['split'] == 'train'].reset_index(drop=True)
val_df   = split_df[split_df['split'] == 'val'].reset_index(drop=True)
test_df  = split_df[split_df['split'] == 'test'].reset_index(drop=True)
print(f'Train: {len(train_df)}  |  Val: {len(val_df)}  |  Test: {len(test_df)}')

# Class weights via median frequency balancing (computed from training split)
class_avg = train_df[CLASS_NAMES].mean().values
class_freq = class_avg / class_avg.sum()
median_freq = np.median(class_freq)
class_weights = median_freq / (class_freq + 1e-8)
class_weights_tensor = torch.FloatTensor(class_weights).to(device)
print(f'Class weights: {class_weights.round(2)}')

Train: 7000  |  Val: 1500  |  Test: 1500
Class weights: [0.15 5.12 0.09 0.24 3.77 1.   2.54]


## 2. Dataset and DataLoaders

The Dataset returns (image, mask, caption) tuples. The caption column is a constructor argument so the same class works for any of the 5 caption variants.

In [8]:
# Dataset class — reads pre-converted class-index masks and a caption column
class RSMultiModalDataset(Dataset):
    def __init__(self, dataframe, images_dir, masks_dir, caption_col):
        self.df = dataframe.reset_index(drop=True)
        self.images_dir = Path(images_dir)
        self.masks_dir  = Path(masks_dir)
        self.caption_col = caption_col
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        fname = row['filename']

        img = Image.open(self.images_dir / fname).convert('RGB')
        img = transforms.functional.to_tensor(img)
        img = self.normalize(img)

        mask = np.array(Image.open(self.masks_dir / fname))
        mask = torch.from_numpy(mask).long()

        caption = str(row[self.caption_col])
        return img, mask, caption

In [9]:
BATCH_SIZE  = 8
NUM_WORKERS = 4

def make_loaders(caption_col):
    """Build train/val/test loaders for a given caption column."""
    train_dataset = RSMultiModalDataset(train_df, LOCAL_IMAGES, LOCAL_MASKS_CLASS, caption_col)
    val_dataset   = RSMultiModalDataset(val_df,   LOCAL_IMAGES, LOCAL_MASKS_CLASS, caption_col)
    test_dataset  = RSMultiModalDataset(test_df,  LOCAL_IMAGES, LOCAL_MASKS_CLASS, caption_col)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    val_loader   = DataLoader(val_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)
    test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)
    return train_loader, val_loader, test_loader

# Sanity check with the first caption variant
train_loader, val_loader, test_loader = make_loaders(CAPTION_COLS[0])
print(f'Caption column: {CAPTION_COLS[0]}')
print(f'Train: {len(train_loader.dataset)} samples, {len(train_loader)} batches')

imgs, masks, captions = next(iter(train_loader))
print(f'Batch: images {imgs.shape}, masks {masks.shape}, captions[0]: "{captions[0][:80]}..."')

Caption column: hybrid_gemma3-4b
Train: 7000 samples, 875 batches
Batch: images torch.Size([8, 3, 256, 256]), masks torch.Size([8, 256, 256]), captions[0]: "The image depicts a landscape dominated by extensive grasslands (81%) interspers..."


## 3. UNetFormer architecture

Same UNetFormer as in `03_baselines.ipynb` — CNN encoder (`swsl_resnet18`) with Global-Local Transformer Block decoder, Feature Refinement Head, and auxiliary head for deep supervision. Re-defined here so this notebook is self-contained.

Source: https://github.com/WangLibo1995/GeoSeg

In [10]:
# Building blocks (Conv + BN + ReLU variants)
from einops import rearrange
from timm.models.layers import DropPath, trunc_normal_
import timm


class ConvBNReLU(nn.Sequential):
    def __init__(self, in_channels, out_channels, kernel_size=3, dilation=1, stride=1,
                 norm_layer=nn.BatchNorm2d, bias=False):
        super().__init__(
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, bias=bias,
                      dilation=dilation, stride=stride,
                      padding=((stride - 1) + dilation * (kernel_size - 1)) // 2),
            norm_layer(out_channels), nn.ReLU6())


class ConvBN(nn.Sequential):
    def __init__(self, in_channels, out_channels, kernel_size=3, dilation=1, stride=1,
                 norm_layer=nn.BatchNorm2d, bias=False):
        super().__init__(
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, bias=bias,
                      dilation=dilation, stride=stride,
                      padding=((stride - 1) + dilation * (kernel_size - 1)) // 2),
            norm_layer(out_channels))


class Conv(nn.Sequential):
    def __init__(self, in_channels, out_channels, kernel_size=3, dilation=1, stride=1, bias=False):
        super().__init__(
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, bias=bias,
                      dilation=dilation, stride=stride,
                      padding=((stride - 1) + dilation * (kernel_size - 1)) // 2))


class SeparableConvBN(nn.Sequential):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, dilation=1,
                 norm_layer=nn.BatchNorm2d):
        super().__init__(
            nn.Conv2d(in_channels, in_channels, kernel_size, stride=stride, dilation=dilation,
                      padding=((stride - 1) + dilation * (kernel_size - 1)) // 2,
                      groups=in_channels, bias=False),
            norm_layer(out_channels),
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False))

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [11]:
# MLP and Global-Local Attention (window self-attention + local conv branch)
class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None,
                 act_layer=nn.ReLU6, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Conv2d(in_features, hidden_features, 1, 1, 0, bias=True)
        self.act = act_layer()
        self.fc2 = nn.Conv2d(hidden_features, out_features, 1, 1, 0, bias=True)
        self.drop = nn.Dropout(drop, inplace=True)

    def forward(self, x):
        x = self.fc1(x); x = self.act(x); x = self.drop(x)
        x = self.fc2(x); x = self.drop(x); return x


class GlobalLocalAttention(nn.Module):
    def __init__(self, dim=256, num_heads=16, qkv_bias=False,
                 window_size=8, relative_pos_embedding=True):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // self.num_heads
        self.scale = head_dim ** -0.5
        self.ws = window_size
        self.qkv = Conv(dim, 3 * dim, kernel_size=1, bias=qkv_bias)
        self.local1 = ConvBN(dim, dim, kernel_size=3)
        self.local2 = ConvBN(dim, dim, kernel_size=1)
        self.proj = SeparableConvBN(dim, dim, kernel_size=window_size)
        self.attn_x = nn.AvgPool2d(kernel_size=(window_size, 1), stride=1,
                                    padding=(window_size // 2 - 1, 0))
        self.attn_y = nn.AvgPool2d(kernel_size=(1, window_size), stride=1,
                                    padding=(0, window_size // 2 - 1))
        self.relative_pos_embedding = relative_pos_embedding
        if self.relative_pos_embedding:
            self.relative_position_bias_table = nn.Parameter(
                torch.zeros((2 * window_size - 1) * (2 * window_size - 1), num_heads))
            coords_h = torch.arange(self.ws); coords_w = torch.arange(self.ws)
            coords = torch.stack(torch.meshgrid([coords_h, coords_w]))
            coords_flatten = torch.flatten(coords, 1)
            relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]
            relative_coords = relative_coords.permute(1, 2, 0).contiguous()
            relative_coords[:, :, 0] += self.ws - 1; relative_coords[:, :, 1] += self.ws - 1
            relative_coords[:, :, 0] *= 2 * self.ws - 1
            relative_position_index = relative_coords.sum(-1)
            self.register_buffer("relative_position_index", relative_position_index)
            trunc_normal_(self.relative_position_bias_table, std=.02)

    def pad(self, x, ps):
        _, _, H, W = x.size()
        if W % ps != 0: x = F.pad(x, (0, ps - W % ps), mode='reflect')
        if H % ps != 0: x = F.pad(x, (0, 0, 0, ps - H % ps), mode='reflect')
        return x

    def pad_out(self, x):
        return F.pad(x, pad=(0, 1, 0, 1), mode='reflect')

    def forward(self, x):
        B, C, H, W = x.shape
        local = self.local2(x) + self.local1(x)
        x = self.pad(x, self.ws); B, C, Hp, Wp = x.shape
        qkv = self.qkv(x)
        q, k, v = rearrange(qkv, 'b (qkv h d) (hh ws1) (ww ws2) -> qkv (b hh ww) h (ws1 ws2) d',
            h=self.num_heads, d=C // self.num_heads, hh=Hp // self.ws, ww=Wp // self.ws,
            qkv=3, ws1=self.ws, ws2=self.ws)
        dots = (q @ k.transpose(-2, -1)) * self.scale
        if self.relative_pos_embedding:
            relative_position_bias = self.relative_position_bias_table[
                self.relative_position_index.view(-1)].view(self.ws * self.ws, self.ws * self.ws, -1)
            relative_position_bias = relative_position_bias.permute(2, 0, 1).contiguous()
            dots += relative_position_bias.unsqueeze(0)
        attn = dots.softmax(dim=-1); attn = attn @ v
        attn = rearrange(attn, '(b hh ww) h (ws1 ws2) d -> b (h d) (hh ws1) (ww ws2)',
            h=self.num_heads, d=C // self.num_heads, hh=Hp // self.ws, ww=Wp // self.ws,
            ws1=self.ws, ws2=self.ws)
        attn = attn[:, :, :H, :W]
        out = self.attn_x(F.pad(attn, pad=(0, 0, 0, 1), mode='reflect')) + \
              self.attn_y(F.pad(attn, pad=(0, 1, 0, 0), mode='reflect'))
        out = out + local; out = self.pad_out(out); out = self.proj(out)
        out = out[:, :, :H, :W]
        return out

In [12]:
# Transformer Block (GLTB)
class Block(nn.Module):
    def __init__(self, dim=256, num_heads=16, mlp_ratio=4., qkv_bias=False,
                 drop=0., drop_path=0.,
                 act_layer=nn.ReLU6, norm_layer=nn.BatchNorm2d, window_size=8):
        super().__init__()
        self.norm1 = norm_layer(dim)
        self.attn = GlobalLocalAttention(dim, num_heads=num_heads, qkv_bias=qkv_bias,
                                         window_size=window_size)
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden_dim, out_features=dim,
                       act_layer=act_layer, drop=drop)
        self.norm2 = norm_layer(dim)

    def forward(self, x):
        x = x + self.drop_path(self.attn(self.norm1(x)))
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x

In [13]:
# Decoder components: weighted feature fusion (WF), feature refinement head, aux head
class WF(nn.Module):
    def __init__(self, in_channels=128, decode_channels=128, eps=1e-8):
        super().__init__()
        self.pre_conv = Conv(in_channels, decode_channels, kernel_size=1)
        self.weights = nn.Parameter(torch.ones(2, dtype=torch.float32), requires_grad=True)
        self.eps = eps
        self.post_conv = ConvBNReLU(decode_channels, decode_channels, kernel_size=3)

    def forward(self, x, res):
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        weights = nn.ReLU()(self.weights)
        fuse_weights = weights / (torch.sum(weights, dim=0) + self.eps)
        x = fuse_weights[0] * self.pre_conv(res) + fuse_weights[1] * x
        return self.post_conv(x)


class FeatureRefinementHead(nn.Module):
    def __init__(self, in_channels=64, decode_channels=64):
        super().__init__()
        self.pre_conv = Conv(in_channels, decode_channels, kernel_size=1)
        self.weights = nn.Parameter(torch.ones(2, dtype=torch.float32), requires_grad=True)
        self.eps = 1e-8
        self.post_conv = ConvBNReLU(decode_channels, decode_channels, kernel_size=3)
        self.pa = nn.Sequential(
            nn.Conv2d(decode_channels, decode_channels, kernel_size=3, padding=1,
                      groups=decode_channels), nn.Sigmoid())
        self.ca = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            Conv(decode_channels, decode_channels // 16, kernel_size=1), nn.ReLU6(),
            Conv(decode_channels // 16, decode_channels, kernel_size=1), nn.Sigmoid())
        self.shortcut = ConvBN(decode_channels, decode_channels, kernel_size=1)
        self.proj = SeparableConvBN(decode_channels, decode_channels, kernel_size=3)
        self.act = nn.ReLU6()

    def forward(self, x, res):
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        weights = nn.ReLU()(self.weights)
        fuse_weights = weights / (torch.sum(weights, dim=0) + self.eps)
        x = fuse_weights[0] * self.pre_conv(res) + fuse_weights[1] * x
        x = self.post_conv(x); shortcut = self.shortcut(x)
        pa = self.pa(x) * x; ca = self.ca(x) * x
        x = pa + ca; x = self.proj(x) + shortcut
        return self.act(x)


class AuxHead(nn.Module):
    def __init__(self, in_channels=64, num_classes=8):
        super().__init__()
        self.conv = ConvBNReLU(in_channels, in_channels)
        self.drop = nn.Dropout(0.1)
        self.conv_out = Conv(in_channels, num_classes, kernel_size=1)

    def forward(self, x, h, w):
        feat = self.conv(x); feat = self.drop(feat); feat = self.conv_out(feat)
        return F.interpolate(feat, size=(h, w), mode='bilinear', align_corners=False)

## 4. Multimodal architecture

The multimodal model extends UNetFormer with three additions:

1. **Frozen CLIP text encoder** (`ViT-B-32`, `laion2b_s34b_b79k`): converts each caption into a 512-d L2-normalized embedding. No gradients flow into CLIP.
2. **Text-Visual Cross-Attention**: visual features attend to the text embedding via a gated residual. The gate parameter starts at zero, so text influence ramps up gradually during training instead of disrupting the pretrained backbone.
3. **TextAugmentedDecoder**: same as the UNetFormer decoder, but with a cross-attention module after each Global-Local Transformer Block.

Only the cross-attention modules and the decoder are trained. The CNN encoder (`swsl_resnet18`) and CLIP text encoder are both frozen — same setup as PoC.

In [19]:
import open_clip


class CLIPTextEncoder(nn.Module):
    """Frozen CLIP text encoder. Outputs L2-normalized 512-d embeddings."""

    def __init__(self, model_name='ViT-B-32', pretrained='laion2b_s34b_b79k'):
        super().__init__()
        clip_model, _, _ = open_clip.create_model_and_transforms(model_name, pretrained=pretrained)
        self.clip_model = clip_model
        self.tokenizer = open_clip.get_tokenizer(model_name)

        # Drop the visual tower — we only use the text encoder.
        # Saves ~88M frozen params from GPU memory.
        if hasattr(self.clip_model, 'visual'):
            del self.clip_model.visual

        # Freeze all remaining CLIP parameters
        for param in self.parameters():
            param.requires_grad = False

    @torch.no_grad()
    def forward(self, text_list):
        device = next(self.parameters()).device
        tokens = self.tokenizer(text_list).to(device)
        text_features = self.clip_model.encode_text(tokens)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)
        return text_features


# Verify
clip_enc = CLIPTextEncoder().to(device)
with torch.no_grad():
    emb = clip_enc(['test caption'])
    print(f'CLIP text encoder loaded: output {emb.shape}')
    print(f'Frozen params: {sum(p.numel() for p in clip_enc.parameters()) / 1e6:.1f}M')
del clip_enc

CLIP text encoder loaded: output torch.Size([1, 512])
Frozen params: 63.4M


In [20]:
class TextVisualCrossAttention(nn.Module):
    """Visual features attend to text embedding via gated residual.

    The gate starts at 0, meaning text initially has no influence. As training
    progresses, the gate learns to weight the text contribution, allowing the
    backbone to retain its pretrained features at the start.
    """

    def __init__(self, visual_dim=64, text_dim=512, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = visual_dim // num_heads
        self.scale = self.head_dim ** -0.5

        # Project text embedding into the visual feature space
        self.text_proj = nn.Sequential(
            nn.Linear(text_dim, visual_dim), nn.ReLU6(),
            nn.Linear(visual_dim, visual_dim),
        )

        self.q_proj = nn.Conv2d(visual_dim, visual_dim, 1)
        self.k_proj = nn.Linear(visual_dim, visual_dim)
        self.v_proj = nn.Linear(visual_dim, visual_dim)
        self.out_proj = nn.Sequential(
            nn.Conv2d(visual_dim, visual_dim, 1), nn.BatchNorm2d(visual_dim),
        )
        # Gated residual — starts at 0, learns to scale text contribution
        self.gate = nn.Parameter(torch.zeros(1))

    def forward(self, visual, text_embed):
        B, C, H, W = visual.shape
        text_feat = self.text_proj(text_embed)
        q = self.q_proj(visual).reshape(B, self.num_heads, self.head_dim, H * W).permute(0, 1, 3, 2)
        k = self.k_proj(text_feat).reshape(B, self.num_heads, 1, self.head_dim)
        v = self.v_proj(text_feat).reshape(B, self.num_heads, 1, self.head_dim)
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        out = (attn @ v).permute(0, 1, 3, 2).reshape(B, C, H, W)
        out = self.out_proj(out)
        return visual + self.gate * out

In [21]:
class TextAugmentedDecoder(nn.Module):
    """UNetFormer decoder + cross-attention after each GLTB."""

    def __init__(self, encoder_channels=(64, 128, 256, 512), decode_channels=64,
                 dropout=0.1, window_size=8, num_classes=7, text_dim=512):
        super().__init__()
        self.pre_conv = ConvBN(encoder_channels[-1], decode_channels, kernel_size=1)
        self.b4 = Block(dim=decode_channels, num_heads=8, window_size=window_size)
        self.p3 = WF(encoder_channels[-2], decode_channels)
        self.b3 = Block(dim=decode_channels, num_heads=8, window_size=window_size)
        self.p2 = WF(encoder_channels[-3], decode_channels)
        self.b2 = Block(dim=decode_channels, num_heads=8, window_size=window_size)
        self.p1 = FeatureRefinementHead(encoder_channels[-4], decode_channels)
        self.up4 = nn.UpsamplingBilinear2d(scale_factor=4)
        self.up3 = nn.UpsamplingBilinear2d(scale_factor=2)
        self.aux_head = AuxHead(decode_channels, num_classes)
        self.segmentation_head = nn.Sequential(
            ConvBNReLU(decode_channels, decode_channels),
            nn.Dropout2d(p=dropout, inplace=True),
            Conv(decode_channels, num_classes, kernel_size=1),
        )

        # Text-visual cross-attention at each decoder stage
        self.text_attn4 = TextVisualCrossAttention(decode_channels, text_dim, num_heads=4)
        self.text_attn3 = TextVisualCrossAttention(decode_channels, text_dim, num_heads=4)
        self.text_attn2 = TextVisualCrossAttention(decode_channels, text_dim, num_heads=4)
        self.text_attn1 = TextVisualCrossAttention(decode_channels, text_dim, num_heads=4)

        self.init_weight()

    def forward(self, res1, res2, res3, res4, h, w, text_embed):
        if self.training:
            x = self.b4(self.pre_conv(res4)); x = self.text_attn4(x, text_embed); h4 = self.up4(x)
            x = self.p3(x, res3); x = self.b3(x); x = self.text_attn3(x, text_embed); h3 = self.up3(x)
            x = self.p2(x, res2); x = self.b2(x); x = self.text_attn2(x, text_embed); h2 = x
            x = self.p1(x, res1); x = self.text_attn1(x, text_embed)
            x = self.segmentation_head(x)
            x = F.interpolate(x, size=(h, w), mode='bilinear', align_corners=False)
            ah = h4 + h3 + h2; ah = self.aux_head(ah, h, w)
            return x, ah
        else:
            x = self.b4(self.pre_conv(res4)); x = self.text_attn4(x, text_embed)
            x = self.p3(x, res3); x = self.b3(x); x = self.text_attn3(x, text_embed)
            x = self.p2(x, res2); x = self.b2(x); x = self.text_attn2(x, text_embed)
            x = self.p1(x, res1); x = self.text_attn1(x, text_embed)
            x = self.segmentation_head(x)
            return F.interpolate(x, size=(h, w), mode='bilinear', align_corners=False)

    def init_weight(self):
        for m in self.children():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, a=1)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

In [22]:
class UNetFormerCLIP(nn.Module):
    """UNetFormer + frozen CLIP text encoder + cross-attention fusion."""

    def __init__(self, decode_channels=64, dropout=0.1, backbone_name='swsl_resnet18',
                 pretrained=True, window_size=8, num_classes=7,
                 clip_model='ViT-B-32', clip_pretrained='laion2b_s34b_b79k'):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, features_only=True, output_stride=32,
                                          out_indices=(1, 2, 3, 4), pretrained=pretrained)
        encoder_channels = self.backbone.feature_info.channels()
        self.text_encoder = CLIPTextEncoder(clip_model, clip_pretrained)
        self.decoder = TextAugmentedDecoder(encoder_channels, decode_channels, dropout,
                                            window_size, num_classes, text_dim=512)

    def forward(self, x, captions):
        h, w = x.size()[-2:]
        res1, res2, res3, res4 = self.backbone(x)
        text_embed = self.text_encoder(captions)
        if self.training:
            out, ah = self.decoder(res1, res2, res3, res4, h, w, text_embed)
            return out, ah
        else:
            return self.decoder(res1, res2, res3, res4, h, w, text_embed)

In [23]:
# Quick sanity check: instantiate, count parameters, forward pass in both modes
test_model = UNetFormerCLIP(num_classes=NUM_CLASSES).to(device)

total_params     = sum(p.numel() for p in test_model.parameters())
trainable_params = sum(p.numel() for p in test_model.parameters() if p.requires_grad)
frozen_params    = total_params - trainable_params
print(f'Total parameters    : {total_params / 1e6:.2f}M')
print(f'Trainable parameters: {trainable_params / 1e6:.2f}M')
print(f'Frozen parameters   : {frozen_params / 1e6:.2f}M (CLIP text encoder)')

with torch.no_grad():
    sample_imgs, _, sample_caps = next(iter(train_loader))
    sample_imgs = sample_imgs[:2].to(device)
    sample_caps = list(sample_caps[:2])

    test_model.eval()
    out_eval = test_model(sample_imgs, sample_caps)
    print(f'\nEval  mode: input {sample_imgs.shape} -> output {out_eval.shape}')

    test_model.train()
    out_main, out_aux = test_model(sample_imgs, sample_caps)
    print(f'Train mode: main {out_main.shape}, aux {out_aux.shape}')

del test_model
torch.cuda.empty_cache()

Total parameters    : 75.37M
Trainable parameters: 11.94M
Frozen parameters   : 63.43M (CLIP text encoder)

Eval  mode: input torch.Size([2, 3, 256, 256]) -> output torch.Size([2, 7, 256, 256])
Train mode: main torch.Size([2, 7, 256, 256]), aux torch.Size([2, 7, 256, 256])
